# Mock 6A · Molecule Identification from Mass Spectra · Nhận dạng phân tử từ phổ khối (Enveda CASMI 2026 style)

| | 🇬🇧 English | 🇻🇳 Tiếng Việt |
|---|---|---|
| **Task** | A tandem mass spectrometer (MS/MS) breaks a molecule into fragments and measures their masses (m/z) and intensities. Each test row is **one real MS/MS spectrum** of an unknown molecule. For each spectrum, **rank the candidate structures** given for its `molecule_id` (best guess first, up to 25). | Máy khối phổ hai lần (MS/MS) bẻ phân tử thành các mảnh và đo khối lượng (m/z) cùng cường độ của chúng. Mỗi dòng test là **một phổ MS/MS thật** của một phân tử chưa biết. Với mỗi phổ, hãy **xếp hạng các cấu trúc ứng viên** của `molecule_id` đó (đoán tốt nhất trước, tối đa 25). |
| **Data** | `train/` 213,548 spectra of 25,913 molecules **with their structures** (`fold` = `train` / `val`) + `val_candidates.csv` (candidate lists for 1,500 val molecules, to measure MRR@25 yourself) · `public_test/` 5,689 spectra of 1,498 molecules · `private_test/` 4,799 spectra of 1,498 molecules, each molecule with a list of ~128 candidate structures. **No test molecule appears in train**, and the private molecules are the ones least similar to train. | `train/` 213.548 phổ của 25.913 phân tử **kèm cấu trúc** (`fold` = `train` / `val`) + `val_candidates.csv` (danh sách ứng viên cho 1.500 phân tử val, để tự đo MRR@25) · `public_test/` 5.689 phổ của 1.498 phân tử · `private_test/` 4.799 phổ của 1.498 phân tử, mỗi phân tử có danh sách ~128 cấu trúc ứng viên. **Không phân tử test nào có trong train**, và các phân tử private là những phân tử ít giống train nhất. |
| **Metric** | MRR@25 (mean reciprocal rank): a spectrum scores 1/rank of the correct candidate if it is among your first 25 guesses, else 0; mean over all spectra, **higher is better**. Correct = same InChIKey first block (atom connectivity, like CASMI: stereo and tautomers do not matter). Ranking uses the **private** score. | MRR@25: mỗi phổ được 1/hạng của ứng viên đúng nếu nó nằm trong 25 dự đoán đầu, ngược lại 0; lấy trung bình trên mọi phổ, **càng cao càng tốt**. Đúng = cùng khối đầu của InChIKey (cách các nguyên tử nối với nhau, giống CASMI: không xét lập thể và hỗ biến). Xếp hạng theo điểm **private**. |
| **Submission** | `public_submission.csv`, `private_submission.csv` with columns `id,cand_ids`: one row per test spectrum, `cand_ids` = up to 25 `cand_id`s from **that spectrum's** candidate list, best first, separated by `;` (e.g. `S1234567,100231;100198;100250`). | `public_submission.csv`, `private_submission.csv` gồm cột `id,cand_ids`: mỗi phổ test một dòng, `cand_ids` = tối đa 25 `cand_id` lấy từ danh sách ứng viên **của chính phổ đó**, tốt nhất trước, cách nhau bởi `;` (vd. `S1234567,100231;100198;100250`). |
| **Rules** | Any method and any open-source library: **RDKit** (installed by this notebook), matchms, scikit-learn, PyTorch... Use **only the provided files**: no external spectra or structure databases, no pretrained spectrum / molecule models (DreaMS, MIST, ChemBERTa...), no LLM APIs. The data are public (MassSpecGym): **do not look up the test answers** (honour code). Every answer must come from your code. | Được dùng mọi phương pháp và thư viện mã nguồn mở: **RDKit** (notebook tự cài), matchms, scikit-learn, PyTorch... **Chỉ dùng các file được cung cấp**: không dùng phổ hay cơ sở dữ liệu cấu trúc bên ngoài, không dùng mô hình phổ / phân tử huấn luyện sẵn (DreaMS, MIST, ChemBERTa...), không gọi API LLM. Dữ liệu là dữ liệu công khai (MassSpecGym): **không tra đáp án test** (giữ danh dự). Mọi đáp án phải do code của bạn sinh ra. |

$$\text{MRR@25}=\frac{1}{N}\sum_{i=1}^{N}\frac{\mathbb{1}[\text{rank}_i\le 25]}{\text{rank}_i}$$

**Files · Các file:**
- `*.parquet` (spectra · phổ): `id` spectrum id · mã phổ; `molecule_id` (spectra with the same `molecule_id` come from **the same molecule**, as in CASMI · các phổ cùng `molecule_id` là của **cùng một phân tử**, giống CASMI); `mzs`, `intensities` (aligned peak arrays, base peak = 1 · mảng đỉnh song song, đỉnh cao nhất = 1); `precursor_mz` (m/z of the whole ion · m/z của cả ion); `adduct` (`[M+H]+` or `[M+Na]+`); `instrument_type` (`Orbitrap` / `QTOF`, sometimes missing · đôi khi thiếu); `collision_energy` (missing for ~45% · thiếu ở ~45%). Train only · Chỉ train: `fold`, `smiles`, `inchikey14`, `formula`.
- `*_candidates.csv`: `molecule_id, cand_id, smiles` (+ `is_answer` in `val_candidates.csv`, 1 = correct). Candidates are real PubChem molecules with almost the same mass as the answer · Ứng viên là các phân tử thật trong PubChem có khối lượng gần như bằng đáp án.

🇬🇧 This is a real research problem: the best published models on this benchmark find the right molecule first only about 1 time in 7 among 256 candidates. Any steady improvement over the baseline is real progress.
🇻🇳 Đây là bài toán nghiên cứu thật: các mô hình tốt nhất đã công bố trên bộ dữ liệu này chỉ đoán đúng ở vị trí đầu khoảng 1/7 số lần với 256 ứng viên. Mọi cải thiện ổn định so với baseline đều là tiến bộ thật.

Data · Dữ liệu: [MassSpecGym](https://huggingface.co/datasets/roman-bushuiev/MassSpecGym) (Bushuiev et al., NeurIPS 2024), MIT licence (see `dataset/LICENSE`, `dataset/README.md`).
Inspired by · Lấy cảm hứng từ: [Enveda CASMI 2026 – Molecule ID From Mass Spectra (Kaggle)](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra).

🇬🇧 This baseline runs end-to-end and submits a result. Beating it is up to you.
🇻🇳 Baseline này chạy trọn vẹn và nộp được kết quả. Vượt qua nó là việc của bạn.

## 1. Setup and Imports · Cài đặt và thư viện

In [ ]:
import os
import re
import shutil
import unicodedata
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import subprocess
import sys

import scipy.sparse as sp

try:
    import rdkit  # noqa: F401
except ImportError:  # Colab: install RDKit once (~20 s) / cài RDKit một lần
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit"], check=True)
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration and Hyperparameters · Cấu hình và siêu tham số
🇬🇧 Nothing to fill in here: the data downloads itself, and your team comes from your account on the site when you submit (section 8).
🇻🇳 Không cần điền gì ở đây: dữ liệu tự tải về, và đội của bạn được lấy từ tài khoản trên trang web khi nộp bài (phần 8).

In [ ]:
# ---- Data / Dữ liệu ----
# The dataset downloads itself from GitHub straight onto this Colab session's own
# disk. No Google Drive and no mounting: reading thousands of small files over a
# Drive mount is what made the old flow slow, and local disk is several times
# faster per epoch.
# Bộ dữ liệu tự tải từ GitHub vào ổ đĩa của phiên Colab này. Không cần Drive.
TASK    = "6A"
FOLDER  = "6A_mass_spec_molecule_id"
RELEASE = "https://github.com/ftulabs/olympiad-datasets/releases/download/mock-6"

# Already have the folder? Point this at the parent of 6A_mass_spec_molecule_id/ and nothing is
# downloaded. / Đã có sẵn thư mục? Trỏ vào thư mục cha của 6A_mass_spec_molecule_id/.
ROOT_OVERRIDE = ""

import os, subprocess, time

ROOT = ROOT_OVERRIDE or "/content/olympiad"
if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    os.makedirs(ROOT, exist_ok=True)
    zip_path = f"{ROOT}/{FOLDER}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {FOLDER}.zip / Đang tải ...", flush=True)
        t0 = time.time()
        subprocess.run(["curl", "-fsSL", "--retry", "3", "-o", zip_path + ".part",
                        f"{RELEASE}/{FOLDER}.zip"], check=True)
        os.replace(zip_path + ".part", zip_path)  # only a complete download gets the final name
        print(f"  {os.path.getsize(zip_path)/1e6:.0f} MB in {time.time()-t0:.0f}s", flush=True)
    print("Extracting / Đang giải nén ...", flush=True)
    subprocess.run(["unzip", "-nq", zip_path, "-d", ROOT], check=True)

if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    raise FileNotFoundError(
        f"{ROOT}/{FOLDER}/dataset is missing after the download. Re-run this cell. "
        f"/ Thiếu dữ liệu sau khi tải. Hãy chạy lại ô này."
    )
print(f"Data root / Thư mục dữ liệu: {ROOT}")

DATA_DIR = f"{ROOT}/6A_mass_spec_molecule_id/dataset"
TRAIN_PARQUET = f"{DATA_DIR}/train/train.parquet"
VAL_CANDIDATES = f"{DATA_DIR}/train/val_candidates.csv"
PUBLIC_PARQUET = f"{DATA_DIR}/public_test/public_test.parquet"
PUBLIC_CANDIDATES = f"{DATA_DIR}/public_test/public_candidates.csv"
PRIVATE_PARQUET = f"{DATA_DIR}/private_test/private_test.parquet"
PRIVATE_CANDIDATES = f"{DATA_DIR}/private_test/private_candidates.csv"
SAVE_DIR = "./checkpoints"

# ---- Hyperparameters / Siêu tham số ----
MAX_MZ = 1000.0  # peaks above this m/z are dropped / bỏ các đỉnh có m/z lớn hơn
BIN_WIDTH = 1.0  # spectrum -> vector of MAX_MZ / BIN_WIDTH bins / phổ -> vectơ các ô
FP_BITS = 2048  # Morgan fingerprint size / độ dài vân tay Morgan
HIDDEN = 512
BATCH_SIZE = 256
EPOCHS = 5
LR = 1e-3
TOP_K = 25
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs(SAVE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3. Data Loading · Đọc dữ liệu

In [ ]:
train_all = pd.read_parquet(TRAIN_PARQUET)
val_cands = pd.read_csv(VAL_CANDIDATES)
public_df, public_cands = pd.read_parquet(PUBLIC_PARQUET), pd.read_csv(PUBLIC_CANDIDATES)
private_df, private_cands = pd.read_parquet(PRIVATE_PARQUET), pd.read_csv(PRIVATE_CANDIDATES)

for name, df, cands in [("Train+val", train_all, val_cands), ("Public test", public_df, public_cands),
                        ("Private test", private_df, private_cands)]:
    print(f"{name:12s}: {len(df):7,d} spectra, {df['molecule_id'].nunique():6,d} molecules, "
          f"{len(cands):7,d} candidates")
print(train_all["fold"].value_counts().to_dict(), train_all["adduct"].value_counts().to_dict())
train_all.head(3)

## 4. Features: Binned Spectra and Fingerprints · Đặc trưng: phổ chia ô và vân tay phân tử

In [ ]:
N_BINS = int(MAX_MZ / BIN_WIDTH)


def bin_spectra(df):
    """EN: each spectrum -> sparse row of N_BINS bins (sum of sqrt-intensities per bin).
    VI: mỗi phổ -> một hàng thưa N_BINS ô (tổng căn bậc hai cường độ trong ô)."""
    rows, cols, vals = [], [], []
    for i, (mz, inten) in enumerate(zip(df["mzs"], df["intensities"])):
        b = (np.asarray(mz) / BIN_WIDTH).astype(int)
        keep = b < N_BINS
        rows.append(np.full(keep.sum(), i))
        cols.append(b[keep])
        vals.append(np.sqrt(np.asarray(inten, dtype=np.float32)[keep]))
    X = sp.coo_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
                      shape=(len(df), N_BINS)).tocsr()
    X.sum_duplicates()  # several peaks in one bin: keep the sum / nhiều đỉnh cùng ô: cộng lại
    return X


FP_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=FP_BITS)


def fingerprints(smiles_list):
    """EN: SMILES -> Morgan fingerprint bits (n x FP_BITS, uint8). VI: SMILES -> vân tay Morgan."""
    out = np.zeros((len(smiles_list), FP_BITS), dtype=np.uint8)
    for i, s in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(s)
        if mol is not None:
            out[i] = FP_GEN.GetFingerprintAsNumPy(mol)
    return out


# EN: the model learns spectrum -> fingerprint of its molecule; validate on the `val` fold (other molecules)
# VI: mô hình học phổ -> vân tay của phân tử; kiểm tra trên fold `val` (các phân tử khác)
tr_df = train_all[train_all["fold"] == "train"].reset_index(drop=True)
val_df = train_all[train_all["molecule_id"].isin(val_cands["molecule_id"])].reset_index(drop=True)
X_tr, X_val = bin_spectra(tr_df), bin_spectra(val_df)
X_pub, X_priv = bin_spectra(public_df), bin_spectra(private_df)

mol_smiles = tr_df.drop_duplicates("molecule_id").set_index("molecule_id")["smiles"]
mol_fp = fingerprints(mol_smiles.tolist())
mol_index = pd.Series(np.arange(len(mol_smiles)), index=mol_smiles.index)
y_idx = mol_index.loc[tr_df["molecule_id"]].to_numpy()  # row -> its molecule's fingerprint
print(f"Train spectra {X_tr.shape}, molecules {len(mol_fp):,}, mean bits on {mol_fp.sum(1).mean():.1f}")


def unit_fingerprints(cands):
    """EN: candidate fingerprints scaled to length 1 (for cosine). VI: vân tay ứng viên chuẩn hoá độ dài 1."""
    f = fingerprints(cands["smiles"].tolist()).astype(np.float32)
    return f / (np.linalg.norm(f, axis=1, keepdims=True) + 1e-9)


val_cfp, pub_cfp, priv_cfp = unit_fingerprints(val_cands), unit_fingerprints(public_cands), unit_fingerprints(private_cands)

## 5. Model Architecture · Kiến trúc mô hình

In [ ]:
class SpecToFingerprint(nn.Module):
    """EN: binned spectrum -> MLP -> FP_BITS logits (one per fingerprint bit).
    VI: phổ đã chia ô -> MLP -> FP_BITS logit (mỗi bit vân tay một logit)."""

    def __init__(self, n_in=N_BINS, hidden=HIDDEN, n_out=FP_BITS):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden, n_out))

    def forward(self, x):
        return self.net(x)


model = SpecToFingerprint().to(DEVICE)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. Training and Validation (MRR@25 on val) · Huấn luyện và kiểm tra

In [ ]:
def dense(X, idx):
    return torch.from_numpy(X[idx].toarray()).to(DEVICE)


@torch.no_grad()
def predict_fp(model, X):
    """EN: predicted bit probabilities for every spectrum. VI: xác suất từng bit cho mỗi phổ."""
    model.eval()
    out = [torch.sigmoid(model(dense(X, np.arange(i, min(i + 2048, X.shape[0]))))).cpu().numpy()
           for i in range(0, X.shape[0], 2048)]
    return np.concatenate(out)


def rank_candidates(df, pred, cands, cand_fp):
    """EN: for each spectrum, sort its molecule's candidates by cosine(predicted fp, candidate fp); keep TOP_K.
    VI: với mỗi phổ, xếp các ứng viên của phân tử theo cosine(vân tay dự đoán, vân tay ứng viên); giữ TOP_K."""
    groups = cands.groupby("molecule_id").indices
    out = []
    for i, m in enumerate(df["molecule_id"]):
        rows = groups[m]
        score = cand_fp[rows] @ pred[i]
        best = rows[np.argsort(-score, kind="stable")[:TOP_K]]
        out.append(cands["cand_id"].to_numpy()[best].tolist())
    return out


def mrr_at_25(rankings, cands):
    """EN: the competition metric on val: `is_answer` marks the correct candidate of each val molecule.
    VI: độ đo của cuộc thi trên val: `is_answer` đánh dấu ứng viên đúng của mỗi phân tử val."""
    answer = set(cands.loc[cands["is_answer"] == 1, "cand_id"])
    rr = []
    for ranked in rankings:
        hits = [r for r, c in enumerate(ranked[:25], 1) if c in answer]
        rr.append(1.0 / hits[0] if hits else 0.0)
    return float(np.mean(rr))


criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
mol_fp_t = torch.from_numpy(mol_fp.astype(np.float32))

best_mrr = -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    perm, total_loss = np.random.permutation(X_tr.shape[0]), 0.0
    for i in range(0, len(perm), BATCH_SIZE):
        idx = perm[i:i + BATCH_SIZE]
        xb, yb = dense(X_tr, idx), mol_fp_t[y_idx[idx]].to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(idx)

    val_mrr = mrr_at_25(rank_candidates(val_df, predict_fp(model, X_val), val_cands, val_cfp), val_cands)
    msg = f"Epoch {epoch:02d} | Loss={total_loss / len(perm):.4f} | Val MRR@25={val_mrr:.4f}"
    if val_mrr > best_mrr:
        best_mrr = val_mrr
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pt"))
        msg += "  ✓ best model saved"
    print(msg)

print(f"\nBest validation MRR@25: {best_mrr:.4f}")

## 7. Predict Public & Private Test · Dự đoán tập Public & Private
🇬🇧 Creates `public_submission.csv` / `private_submission.csv` (+ `.zip`), like the official competition.
🇻🇳 Tạo `public_submission.csv` / `private_submission.csv` (+ `.zip`), giống định dạng cuộc thi chính thức.

In [ ]:
def export(df, rankings, csv_path):
    """Save <id, cand_ids> CSV and zip it. / Lưu dự đoán ra CSV và nén zip."""
    pd.DataFrame({"id": df["id"], "cand_ids": [";".join(map(str, r)) for r in rankings]}).to_csv(csv_path, index=False)
    zip_path = csv_path.replace(".csv", ".zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=os.path.basename(csv_path))
    print(f"Saved {csv_path} and {zip_path}")


model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best_model.pt"), map_location=DEVICE))
export(public_df, rank_candidates(public_df, predict_fp(model, X_pub), public_cands, pub_cfp), "public_submission.csv")
export(private_df, rank_candidates(private_df, predict_fp(model, X_priv), private_cands, priv_cfp), "private_submission.csv")
pd.read_csv("public_submission.csv").head()

## 8. Submit · Nộp bài

🇬🇧 Your team comes from your account on [seleccion.ftds.online](https://seleccion.ftds.online/), not from a name typed here. The cell below prints a short code once per Colab session; type it on the site and this notebook submits as you.

🇻🇳 Đội của bạn lấy từ tài khoản trên [seleccion.ftds.online](https://seleccion.ftds.online/). Ô bên dưới in ra một mã ngắn mỗi phiên Colab; nhập mã đó trên trang web.

In [ ]:
def check_submission(csv_path, test_df, cands):
    """Validate the file format before submitting. / Kiểm tra định dạng file trước khi nộp."""
    sub = pd.read_csv(csv_path, dtype={"id": str, "cand_ids": str})
    assert list(sub.columns) == ["id", "cand_ids"], f"{csv_path}: columns must be ['id', 'cand_ids']"
    assert len(sub) == len(test_df) and set(sub["id"]) == set(test_df["id"]), f"{csv_path}: ids do not match the test set"
    allowed = cands.groupby("molecule_id")["cand_id"].apply(set)
    mol = dict(zip(test_df["id"], test_df["molecule_id"]))
    for rid, ranked in zip(sub["id"], sub["cand_ids"]):
        tokens = str(ranked).split(";")
        assert all(t.strip().isdigit() for t in tokens), f"{csv_path} {rid}: cand_ids must be integers separated by ';' / mã phải là số nguyên, cách nhau bởi ';'"
        ids = [int(t) for t in tokens]
        assert 1 <= len(ids) <= 25 and len(set(ids)) == len(ids), f"{csv_path} {rid}: 1-25 distinct ids / 1-25 mã khác nhau"
        assert set(ids) <= allowed[mol[rid]], f"{csv_path} {rid}: ids not in this molecule's candidates / mã không thuộc danh sách ứng viên"


check_submission("public_submission.csv", public_df, public_cands)
check_submission("private_submission.csv", private_df, private_cands)
print("Format OK / Định dạng hợp lệ")

In [ ]:
# ===================================================================
#  Olympiad AI - submit.  Paste at the END of your notebook, then run.
#  No key to copy: the first run prints a code you enter on the site.
# ===================================================================
BASE    = "https://seleccion.ftds.online"
PROBLEM = "6A"          # which problem this notebook is for
CONTEST = "mock-6"    # this notebook is for Mock Contest 6
PUBLIC  = "public_submission.csv"
PRIVATE = "private_submission.csv"
# -------------------------------------------------------------------
import os, time, requests

token = globals().get("_OLY_TOKEN")
if not token:
    start = requests.post(f"{BASE}/api/pair/start", timeout=30).json()
    print(f"\n  1. open   {start['enter_at']}")
    print(f"  2. enter  {start['code']}\n  waiting...", flush=True)
    for _ in range(int(start["expires_in"]) // 3):
        time.sleep(3)
        got = requests.get(f"{BASE}/api/pair/poll",
                           params={"code": start["code"]}, timeout=30).json()
        if got.get("ready"):
            token = globals()["_OLY_TOKEN"] = got["token"]
            print("  paired ✓ (this notebook stays paired for the session)")
            break
    else:
        raise SystemExit("pairing timed out - run this cell again")

missing = [f for f in (PUBLIC, PRIVATE) if not os.path.exists(f)]
if PUBLIC in missing:
    raise SystemExit(f"{PUBLIC} not found - run the cell that writes it first")

files = {"public": open(PUBLIC, "rb")}
if PRIVATE not in missing:
    files["private"] = open(PRIVATE, "rb")
try:
    r = requests.post(f"{BASE}/api/submit", headers={"X-Oly-Token": token},
                      data={"problem": PROBLEM, "contest": CONTEST},
                      files=files, timeout=600)
finally:
    for f in files.values():
        f.close()

try:
    d = r.json()
except ValueError:
    raise SystemExit(f"server {r.status_code}: {r.text[:300]}")

if r.status_code == 429:
    print(f"⏳ cooldown - wait {d.get('retry_after')}s and run again")
elif r.status_code == 401:
    globals().pop("_OLY_TOKEN", None)
    print(f"❌ {d.get('detail')}  (run this cell again to re-pair)")
elif r.status_code != 200:
    print(f"❌ {r.status_code}: {d.get('detail') or d.get('message')}")
elif d["public"]["status"] != "ok":
    # the grader's own message about your file - it says what to fix
    print(f"❌ {d['public']['status'].upper()}: {d['public']['message']}")
else:
    p, c = d["public"], (d.get("counting") or {})
    print(f"✅ {d['problem']} · {d['title']}")
    print(f"   {d['metric']} = {p['score']:.4f}   {p['tier'] or '(below the lowest tier)'}")
    print(f"   entry #{d['entries']}   ·   rank: {d['rank']}")
    if c.get("score") is not None and abs(c["score"] - p["score"]) > 1e-12:
        print(f"   counting entry: {c['score']:.4f} {c['tier']}")
    print(f"   {BASE}/")